# HAM10000 — Stage 3: Baseline CNN

**Goal:** Train a CNN from scratch (no pretrained weights) as a baseline.
This gives us a lower-bound to beat with transfer learning in Stage 4.

**Architecture:** 4× ConvBlock (Conv→BN→ReLU→MaxPool) → GAP → Dropout(0.5) → Linear(7)  
**Parameters:** ~1.2M  
**Optimizer:** Adam lr=1e-3, weight_decay=1e-4  
**Loss:** CrossEntropyLoss with class weights from Stage 2  
**Checkpoint:** saved by lowest val loss (not accuracy — misleading under imbalance)  
**Early stopping:** patience=7 epochs

⏱ **Expected runtime on Colab free T4:** 30–60 min (up to 50 epochs, ~1 min/epoch)

## Cell 1 — Clone / update repo and set up paths

In [ ]:
import os, sys, subprocess

REPO_URL  = "https://github.com/Dev252001/HAM10000.git"
REPO_DIR  = "/content/ham10000-classifier"
SRC_DIR   = os.path.join(REPO_DIR, "src")
DATA_DIR  = os.path.join(REPO_DIR, "data")
OUT_DIR   = os.path.join(REPO_DIR, "outputs")

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Repo cloned.")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    print("Repo updated.")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f"src/ on path: {SRC_DIR}")

## Cell 2 — Install dependencies

In [ ]:
!pip install -q \
  "numpy>=2.0" \
  "pandas>=2.2.2" \
  "Pillow>=10.4.0" \
  "scikit-learn>=1.5.0" \
  "matplotlib>=3.9.0" \
  "seaborn>=0.13.2" \
  "kaggle>=1.6.14" \
  "ipywidgets>=8.1.3"
print("Dependencies ready.")

## Cell 3 — Verify GPU

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU found — training will be very slow. Go to Runtime → Change runtime type → GPU.")

## Cell 4 — Load data + build splits and DataLoaders

Reuses Stage 2 preprocessing exactly — same splits (random_state=42), same transforms.

In [ ]:
from data_loader import load_metadata
from preprocessing import make_splits, make_dataloaders, compute_class_weights

df = load_metadata(DATA_DIR)
train_df, val_df, test_df = make_splits(df)

loaders      = make_dataloaders(train_df, val_df, test_df, batch_size=32, num_workers=2)
class_weights = compute_class_weights(train_df)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Class weights: {class_weights.tolist()}")

## Cell 5 — Instantiate the baseline CNN and inspect architecture

In [ ]:
from models.baseline_cnn import build_baseline_cnn

model = build_baseline_cnn(num_classes=7, dropout_p=0.5)

# Count trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model architecture:\n{model}")
print(f"\nTrainable parameters: {total_params:,} (~{total_params/1e6:.2f}M)")

# Sanity check: one forward pass
model.eval()
dummy = torch.zeros(2, 3, 224, 224)
with torch.no_grad():
    out = model(dummy)
print(f"Output shape for batch of 2: {out.shape}  (expected: torch.Size([2, 7]))")

## Cell 6 — Train

⏱ **Expected time:** ~1 min/epoch on T4. With early stopping at patience=7,
expect 15–40 epochs total (~15–40 min).

**Colab session timeout note:** Colab free tier disconnects after ~90 min idle
or ~12 hrs total. The checkpoint is saved after every improvement, so if
the session drops you can reload from `outputs/models/baseline_cnn_best.pt`
and continue evaluation without retraining.

In [ ]:
import os
from train import train

CHECKPOINT = os.path.join(OUT_DIR, "models", "baseline_cnn_best.pt")

model = build_baseline_cnn(num_classes=7, dropout_p=0.5)

history = train(
    model          = model,
    train_loader   = loaders["train"],
    val_loader     = loaders["val"],
    class_weights  = class_weights,
    device         = device,
    num_epochs     = 50,
    lr             = 1e-3,
    weight_decay   = 1e-4,
    early_stopping_patience = 7,
    checkpoint_path = CHECKPOINT,
)

print(f"\nCheckpoint saved to: {CHECKPOINT}")

## Cell 7 — Plot learning curves

In [ ]:
import matplotlib.pyplot as plt

epochs_ran = len(history["train_loss"])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history["train_loss"], label="Train loss")
ax1.plot(history["val_loss"],   label="Val loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss curves")
ax1.legend()

ax2.plot([a * 100 for a in history["train_acc"]], label="Train acc")
ax2.plot([a * 100 for a in history["val_acc"]],   label="Val acc")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Accuracy curves")
ax2.legend()

plt.tight_layout()
fig_path = os.path.join(OUT_DIR, "figures", "baseline_learning_curves.png")
os.makedirs(os.path.dirname(fig_path), exist_ok=True)
plt.savefig(fig_path, dpi=100, bbox_inches="tight")
plt.show()
print(f"Saved → {fig_path}")
print(f"Total epochs trained: {epochs_ran}")

## Cell 8 — Evaluate on test set

Loads the best checkpoint (lowest val loss) and evaluates on the held-out test set.
Malignant-class recall is printed prominently.

In [ ]:
from evaluate import evaluate_model, print_results

# Load best checkpoint
model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
model = model.to(device)

results = evaluate_model(model, loaders["test"], device)
print_results(results)

## Cell 9 — Confusion matrix

In [ ]:
from evaluate import plot_confusion_matrix

cm_path = os.path.join(OUT_DIR, "figures", "baseline_confusion_matrix.png")
plot_confusion_matrix(results["confusion_matrix"], save_path=cm_path)

## Stage 3 complete ✓

**Before moving to Stage 4, verify all boxes below:**

- [ ] Training ran without errors; checkpoint saved to `outputs/models/baseline_cnn_best.pt`
- [ ] Learning curves show loss decreasing — if val loss starts rising while train loss falls, overfitting is occurring (expected for a from-scratch model on 10k images)
- [ ] Overall accuracy is printed — if >90% AND macro-F1 <0.50, the model is likely predicting mostly `nv` — check confusion matrix diagonal
- [ ] Malignant-class recall (mel, bcc, akiec) is printed — reasonable baseline is 0.3–0.6 for a from-scratch model
- [ ] Confusion matrix shows predictions spread across classes (not all in `nv` column)

**Share the accuracy, macro-F1, and malignant-class recall numbers before continuing to Stage 4.**

---

### Design decisions summary

| Decision | Choice | Reason |
|---|---|---|
| Architecture | 4× ConvBlock → GAP → Linear | Deep enough for texture features; shallow enough to avoid overfitting on 7k images |
| Filter progression | 32→64→128→256 | Low-level→high-level features; doubling is standard (VGG convention) |
| GAP vs Flatten+FC | GAP | 256 vs 50k parameters before classifier; reduces overfitting |
| Optimizer | Adam lr=1e-3 | More forgiving than SGD for small imbalanced datasets |
| Loss | CrossEntropyLoss(weight=) | Class weights from Stage 2 force the model to learn minority classes |
| Checkpoint metric | val loss | Accuracy misleading under 67% class imbalance |
| Early stopping | patience=7 | 2 LR reductions before stopping; respects Colab's ~4hr free tier |
| Dropout | 0.5 before classifier | Primary regularisation point for a small-data from-scratch model |

---

### Likely interview questions on this stage

**Q1: Why save the best model by val loss instead of val accuracy?**  
With 67% of images being `nv`, a model that predicts `nv` for everything achieves 67% accuracy in epoch 1 — higher than a model actually learning minority classes. Val loss (with class weights applied) penalises wrong predictions on rare classes more heavily, making it a more honest signal of learning progress.

**Q2: Why Adam over SGD, and what's the trade-off?**  
Adam adapts the learning rate per parameter, which makes it robust to the choice of initial LR — important for small-data training where you don't want to spend epochs tuning the LR. The trade-off is that Adam can sometimes generalise slightly worse than well-tuned SGD+momentum on very large datasets (e.g. ImageNet-scale training), but for ~7k images the robustness benefit outweighs this.

**Q3: The baseline is expected to perform worse than transfer learning — what specifically do you expect to see, and why?**  
From-scratch training on 10k dermoscopic images means the early conv layers must learn low-level feature detectors (edges, colour gradients) from scratch — this takes many samples to converge. A pretrained backbone (Stage 4) already has those detectors from ImageNet, so it can use all available capacity for the HAM10000-specific high-level patterns. I'd expect lower malignant-class recall and lower macro-F1 for the baseline, especially on the smallest classes (df: 115, vasc: 142).